# Notebook 3: Imputation Pipeline
---
Runs the full MICE imputation pipeline to handle missing physical attributes and saves encoder artifacts.

**Prerequisites:**
- NB01 completed (CLIP `.pt` files in `data/processed/outfits/`)
- Raw labels in `data/raw/Outfit4You/label/`

**Outputs:**
- `data/processed/train_imputed_manifest.json`
- `data/processed/val_imputed_manifest.json`
- `data/processed/val_test_outfit_ids.json`
- `saved_models/encoder_mapping.json`
- `saved_models/phys_feature_cols.json`


## Step 1: Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings('ignore')

from scripts.config import (
    DATA_RAW_DIR, DATA_PROCESSED_DIR, SAVED_MODELS_DIR,
    TRAIN_MANIFEST, VAL_MANIFEST,
    TRAIN_IMPUTED_MANIFEST, VAL_IMPUTED_MANIFEST,
)

print("Project root:", Path("..").resolve())
print("Processed dir:", DATA_PROCESSED_DIR)
print("Saved models dir:", SAVED_MODELS_DIR)

Project root: /Users/hazron/O4U
Processed dir: /Users/hazron/O4U/data/processed
Saved models dir: /Users/hazron/O4U/saved_models


## Step 2: Run the Imputation Pipeline

The pipeline:
1. Lowercases all categorical values
2. Binarizes `body_figure` with `MultiLabelBinarizer`
3. Imputes `color_contrast` and `skin_color` via group mode
4. Imputes `hair_style` and `hair_color` with 'unknown' fallback
5. Imputes `height` and `breasts` via MICE (RandomForestClassifier)
6. Adds `_was_imputed` indicator columns for all 6 imputed features
7. One-hot encodes categorical columns on combined train+val
8. Saves `encoder_mapping.json` and `phys_feature_cols.json`
9. Saves `val_test_outfit_ids.json` (for inference safety)

In [2]:
from scripts.o4u_imputation_pipeline import run_pipeline

run_pipeline(
    train_raw=str(TRAIN_MANIFEST),
    val_raw=str(VAL_MANIFEST),
    train_manifest=str(DATA_PROCESSED_DIR / "train_manifest.json"),
    val_manifest=str(DATA_PROCESSED_DIR / "val_manifest.json"),
    train_out=str(TRAIN_IMPUTED_MANIFEST),
    val_out=str(VAL_IMPUTED_MANIFEST),
    n_imputations=1,
)
print("\nImputation pipeline complete.")

🚀 Starting O4U Imputation Pipeline (Merging with Manifest)...
  → Loading Manifests: /Users/hazron/O4U/data/processed/train_manifest.json

[STEP 0] Collecting val + test outfit IDs
  → Loading test raw: /Users/hazron/O4U/data/raw/Outfit4You/label/test.json
  ✓ Saved val_test_outfit_ids.json (4094 IDs) to: /Users/hazron/O4U/data/processed/val_test_outfit_ids.json
  → Loading Raw: /Users/hazron/O4U/data/raw/Outfit4You/label/train.json
  → Loading Raw: /Users/hazron/O4U/data/raw/Outfit4You/label/val.json
  ✓ Added _was_imputed indicators for: ['height', 'breasts', 'skin_color', 'color_contrast', 'hair_style', 'hair_color']

[STEP 2] Processing body_figure (Multi-Label Binarize)
  ✓ Found 11 unique body figure classes

[STEP 3-4] Imputing color_contrast and skin_color
  → Imputing color_contrast via group mode
  → Imputing skin_color via group mode

[STEP 5] Imputing hair features

[STEP 6] MICE Imputation for height & breasts (seed=42)
  ✓ Saved encoder_mapping.json to: /Users/hazron/O4U/

## Step 3: Verify Imputed Manifests

In [3]:
import pandas as pd

df_train = pd.read_json(TRAIN_IMPUTED_MANIFEST)
df_val   = pd.read_json(VAL_IMPUTED_MANIFEST)

print(f"Train manifest: {len(df_train):,} rows, {df_train.shape[1]} columns")
print(f"Val manifest  : {len(df_val):,} rows, {df_val.shape[1]} columns")

# Show imputation indicator columns
indicator_cols = [c for c in df_train.columns if c.endswith('_was_imputed')]
print(f"\nImputation indicators: {indicator_cols}")
print(df_train[indicator_cols].mean().round(3).to_string())

Train manifest: 10,080 rows, 108 columns
Val manifest  : 2,520 rows, 108 columns

Imputation indicators: ['height_was_imputed', 'breasts_was_imputed', 'skin_color_was_imputed', 'color_contrast_was_imputed', 'hair_style_was_imputed', 'hair_color_was_imputed']
height_was_imputed            0.716
breasts_was_imputed           0.698
skin_color_was_imputed        0.725
color_contrast_was_imputed    0.863
hair_style_was_imputed        0.996
hair_color_was_imputed        0.797


## Summary

| Step | Output |
|---|---|
| Imputation pipeline | `train_imputed_manifest.json`, `val_imputed_manifest.json` |
| Encoder artifacts | `encoder_mapping.json`, `phys_feature_cols.json` |

**Next step:** Run NB04 to compute color harmony scores before training.